# V7 — Fixed Calibration + Red Herring Avoidance + IoU Push

**Root cause of V5's RH failure:** Rank averaging spread probabilities
uniformly → 49.7% flagged as mules. Need well-calibrated probs (~2.8% mule rate).

**Fixes:**
1. Use V4's simple average (mean prob = 0.032, not 0.50)
2. Add alert_reason-aware features from train_labels
3. Train a second-stage "is this a red herring?" filter
4. V6's adaptive CDF temporal windows (best IoU)

---

## 0 — Setup

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score, fbeta_score,
                             precision_score, recall_score, confusion_matrix)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from glob import glob
import pyarrow.parquet as pq
import gc, warnings
warnings.filterwarnings('ignore')

DATA_DIR = "IITD-Tryst-Hackathon/Phase 2"
t0 = time.time()

print("Loading data...")
train = pd.read_csv("features_train_p2.csv")
test = pd.read_csv("features_test_p2.csv")
accounts = pd.read_parquet(f"{DATA_DIR}/accounts.parquet")
labels = pd.read_parquet(f"{DATA_DIR}/train_labels.parquet")

train = train.merge(accounts[["account_id", "branch_code"]], on="account_id", how="left")
test = test.merge(accounts[["account_id", "branch_code"]], on="account_id", how="left")
train = train.merge(labels[["account_id", "alert_reason"]], on="account_id", how="left")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Mule rate: {train['is_mule'].mean():.4f} ({train['is_mule'].sum()} mules)")

Loading data...
Train: (96091, 73) | Test: (64062, 71)
Mule rate: 0.0279 (2683 mules)


## 1 — OOF Target Encoding (from Phase3)

In [2]:
print("OOF Target Encoding...")
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train["branch_mule_rate_oof"] = np.nan
global_mean = train["is_mule"].mean()

for tr_idx, val_idx in skf_te.split(train, train["is_mule"]):
    tr_df = train.iloc[tr_idx]
    branch_stats = tr_df.groupby("branch_code")["is_mule"].agg(['sum', 'count'])
    branch_stats["rate"] = (branch_stats["sum"] + 10 * global_mean) / (branch_stats["count"] + 10)
    mapped = train.iloc[val_idx]["branch_code"].map(branch_stats["rate"]).fillna(global_mean)
    train.loc[train.index[val_idx], "branch_mule_rate_oof"] = mapped.values

branch_stats_full = train.groupby("branch_code")["is_mule"].agg(['sum', 'count'])
branch_stats_full["rate"] = (branch_stats_full["sum"] + 10 * global_mean) / (branch_stats_full["count"] + 10)
test["branch_mule_rate_oof"] = test["branch_code"].map(branch_stats_full["rate"]).fillna(global_mean)

score_cols = ["near_threshold_pct", "round_amount_pct", "gap_cv", "degree_centrality",
              "mule_trigram_count", "branch_mule_rate_oof", "has_prior_freeze"]
for df in [train, test]:
    c_score = np.zeros(len(df))
    for col in score_cols:
        if col in df.columns:
            m, s = train[col].mean(), train[col].std()
            c_score += (df[col].fillna(m) - m) / s if s > 0 else 0
    df["composite_score_fixed"] = c_score

OOF Target Encoding...


## 2 — Red Herring Awareness Features

The test set contains red herrings: accounts designed to look like mules
but aren't. We can learn from the train labels which patterns are
more likely to be red herrings.

In [3]:
print("Building Red Herring Awareness features...")

# Map each mule to which of the 13 known patterns triggered it
alert_map = {
    "Dormant Account Reactivation": "dormant",
    "Structuring Transactions Below Threshold": "structuring",
    "Rapid Movement of Funds": "rapid_passthrough",
    "Unusual Fund Flow Pattern": "fan_in_out",
    "Geographic Anomaly Detected": "geo_anomaly",
    "High-Value Activity on New Account": "new_high_value",
    "Income-Transaction Mismatch": "income_mismatch",
    "Post-Contact-Update Spike": "post_mobile",
    "Round Amount Pattern": "round_amounts",
    "Layered Transaction Pattern": "layered",
    "Salary Cycle Anomaly": "salary",
    "Branch Cluster Investigation": "branch_cluster",
    "MCC-Amount Distribution Anomaly": "mcc_anomaly",
    "Routine Investigation": "routine",
}

# Create per-pattern OOF risk scores
# For each alert_reason, compute OOF probability → helps model distinguish real from RH
pattern_cols = []
for reason, col_name in alert_map.items():
    reason_mask = train["alert_reason"] == reason
    if reason_mask.sum() > 10:
        # Accounts flagged for this reason that the model thinks are NOT mules
        # are likely red herrings
        col = f"pattern_{col_name}"
        pattern_cols.append(col)

# Build behavioral features that distinguish RH from real mules
# Key insight: red herrings mimic ONE pattern but lack corroborating signals

for df in [train, test]:
    # Multi-signal score: how many mule patterns does this account match?
    signals = []
    
    # 1. Dormant reactivation: long gap between open_date and first large txn
    if "days_to_first_large" in df.columns and "active_days" in df.columns:
        df["sig_dormant"] = (df["days_to_first_large"] > 180).astype(float)
        signals.append("sig_dormant")
    
    # 2. Structuring: near_threshold_pct > 0.05
    if "near_threshold_pct" in df.columns:
        df["sig_structuring"] = (df["near_threshold_pct"] > df["near_threshold_pct"].quantile(0.90)).astype(float)
        signals.append("sig_structuring")
    
    # 3. Rapid pass-through: low median_dwell_hours
    if "median_dwell_hours" in df.columns:
        df["sig_rapid"] = (df["median_dwell_hours"] < 24).astype(float)
        signals.append("sig_rapid")
    
    # 4. Fan-in/fan-out: high unique_cp_count relative to txn_count
    if "unique_cp_count" in df.columns and "txn_count" in df.columns:
        df["sig_fanout"] = (df["unique_cp_count"] / df["txn_count"].clip(1) > 0.5).astype(float)
        signals.append("sig_fanout")
    
    # 5. New account high value
    if "days_to_first_large" in df.columns and "total_volume" in df.columns:
        df["sig_new_highvol"] = ((df["days_to_first_large"] < 30) & 
                                  (df["total_volume"] > df["total_volume"].quantile(0.75))).astype(float)
        signals.append("sig_new_highvol")
    
    # 6. Round amounts
    if "round_amount_pct" in df.columns:
        df["sig_round"] = (df["round_amount_pct"] > df["round_amount_pct"].quantile(0.90)).astype(float)
        signals.append("sig_round")
    
    # 7. Post-mobile spike
    if "has_mobile_update" in df.columns and "total_volume" in df.columns:
        df["sig_post_mobile"] = (df["has_mobile_update"] == 1).astype(float)
        signals.append("sig_post_mobile")
    
    # 8. Income mismatch: high volume but low balance
    if "total_volume" in df.columns and "balance_mean" in df.columns:
        vol_to_bal = df["total_volume"] / df["balance_mean"].abs().clip(1)
        df["sig_income_mismatch"] = (vol_to_bal > vol_to_bal.quantile(0.90)).astype(float)
        signals.append("sig_income_mismatch")
    
    # 9. Branch cluster
    if "branch_mule_rate_oof" in df.columns:
        df["sig_branch_cluster"] = (df["branch_mule_rate_oof"] > df["branch_mule_rate_oof"].quantile(0.90)).astype(float)
        signals.append("sig_branch_cluster")
    
    # Multi-signal count: real mules match MULTIPLE patterns, RH only match 1
    if signals:
        df["multi_signal_count"] = sum(df[s] for s in signals)
        df["multi_signal_binary"] = (df["multi_signal_count"] >= 3).astype(float)
    
    # Consistency score: ratio of matching patterns to expected patterns
    if signals:
        df["signal_consistency"] = df["multi_signal_count"] / len(signals)

print(f"Added {len(signals)} signal features + multi_signal_count + consistency")

Building Red Herring Awareness features...
Added 7 signal features + multi_signal_count + consistency


## 3 — Prepare Features

In [4]:
drop_cols = ["account_id", "is_mule", "first_large_ts", "open_date",
             "branch_code", "branch_mule_rate", "composite_score", "alert_reason"]
features = [c for c in train.columns if c not in drop_cols and train[c].nunique() > 1]
train[features] = train[features].fillna(train[features].median())
test[features] = test[features].fillna(train[features].median())
print(f"Total features: {len(features)}")

Total features: 75


## 4 — Red Herring Pruning (Aggressive)

In [5]:
print("Red herring pruning...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = train[features].values
y = train["is_mule"].values
oof_screen = np.zeros(len(y))

for fold, (tr, val) in enumerate(skf.split(X, y)):
    m = lgb.LGBMClassifier(n_estimators=300, random_state=42, n_jobs=-1, verbosity=-1)
    m.fit(X[tr], y[tr])
    oof_screen[val] = m.predict_proba(X[val])[:,1]

# Standard pruning: drop extreme red herrings
extreme = (y == 1) & (oof_screen < 0.02)
# Also prune moderate red herrings with low multi_signal_count
moderate_rh = (y == 1) & (oof_screen < 0.10) & (train["multi_signal_count"] < 2)
prune_mask = extreme | moderate_rh

keep_mask = ~prune_mask
X_clean = X[keep_mask]
y_clean = y[keep_mask]

# Soft-weight ambiguous samples instead of dropping
sample_weights = np.ones(len(y_clean))
ambig = ((y == 1) & (oof_screen >= 0.02) & (oof_screen < 0.15) & (train["multi_signal_count"] < 3))[keep_mask]
sample_weights[ambig] = 0.5  # Down-weight suspicious labels

print(f"Pruned {prune_mask.sum()} red herrings ({extreme.sum()} extreme + {(moderate_rh & ~extreme).sum()} moderate)")
print(f"Down-weighted {ambig.sum()} ambiguous samples")
print(f"Training on {len(y_clean):,} samples")

Red herring pruning...
Pruned 414 red herrings (344 extreme + 70 moderate)
Down-weighted 70 ambiguous samples
Training on 95,677 samples


## 5 — Ensemble (Simple Average, Well-Calibrated)

In [6]:
print("Training LGB + XGB + CatBoost (simple average)...")
spw = (y_clean == 0).sum() / max(1, (y_clean == 1).sum())
oof_lgb = np.zeros(len(y_clean))
oof_xgb = np.zeros(len(y_clean))
oof_cat = np.zeros(len(y_clean))
t_lgb = np.zeros(len(test))
t_xgb = np.zeros(len(test))
t_cat = np.zeros(len(test))

X_test = test[features].values

for fold, (tr, val) in enumerate(skf.split(X_clean, y_clean)):
    print(f"--- Fold {fold+1} ---")
    Xtr, Xval = X_clean[tr], X_clean[val]
    ytr, yval = y_clean[tr], y_clean[val]
    wtr = sample_weights[tr]

    m1 = lgb.LGBMClassifier(n_estimators=1200, learning_rate=0.03, max_depth=8,
                            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                            random_state=42, verbosity=-1, n_jobs=-1)
    m1.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xval, yval)],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val] = m1.predict_proba(Xval)[:,1]
    t_lgb += m1.predict_proba(X_test)[:,1] / 5.0

    m2 = xgb.XGBClassifier(n_estimators=1200, learning_rate=0.03, max_depth=7,
                           subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                           random_state=42, verbosity=0, eval_metric="auc", n_jobs=-1,
                           early_stopping_rounds=50)
    m2.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xval, yval)], verbose=False)
    oof_xgb[val] = m2.predict_proba(Xval)[:,1]
    t_xgb += m2.predict_proba(X_test)[:,1] / 5.0

    m3 = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=7,
                            auto_class_weights='Balanced', random_state=42,
                            verbose=False, early_stopping_rounds=50)
    m3.fit(Xtr, ytr, eval_set=(Xval, yval))
    oof_cat[val] = m3.predict_proba(Xval)[:,1]
    t_cat += m3.predict_proba(X_test)[:,1] / 5.0

# Simple average (well-calibrated, not rank)
oof_ens = (oof_lgb + oof_xgb + oof_cat) / 3.0
t_ens = (t_lgb + t_xgb + t_cat) / 3.0

auc = roc_auc_score(y_clean, oof_ens)
print(f"\nOOF AUC: {auc:.4f}")
print(f"Test mean prob: {t_ens.mean():.4f} (should be ~{global_mean:.4f})")
print(f"Test >0.5: {(t_ens>0.5).sum()} (should be ~{int(len(test)*global_mean)})")

Training LGB + XGB + CatBoost (simple average)...
--- Fold 1 ---
--- Fold 2 ---
--- Fold 3 ---
--- Fold 4 ---
--- Fold 5 ---

OOF AUC: 0.9972
Test mean prob: 0.0300 (should be ~0.0279)
Test >0.5: 1743 (should be ~1788)


## 6 — Red Herring Post-Filter

After getting probabilities, apply rule-based checks to down-weight
accounts that match only ONE suspicious pattern (likely planted RH).

In [7]:
print("Applying RH post-filter...")

# Get multi_signal_count for test set
test_signals = test["multi_signal_count"].values

# Scale down test probabilities for low-signal accounts
t_final = t_ens.copy()

# Accounts with high probability but low signal count → likely RH
for i in range(len(t_final)):
    prob = t_final[i]
    signals = test_signals[i]
    
    if prob > 0.3 and signals < 2:
        # High prob but few corroborating signals → suspicious, dampen
        t_final[i] = prob * 0.6
    elif prob > 0.3 and signals < 3:
        # Moderate dampening
        t_final[i] = prob * 0.8

dampened = (t_final != t_ens).sum()
print(f"Dampened {dampened} accounts with high prob but low signal count")
print(f"After filter: >0.5 = {(t_final>0.5).sum()} (was {(t_ens>0.5).sum()})")

Applying RH post-filter...
Dampened 1662 accounts with high prob but low signal count
After filter: >0.5 = 1497 (was 1743)


## 7 — Threshold Optimization

In [8]:
# Optimize on OOF with same filter logic
oof_final = oof_ens.copy()
train_signals = train["multi_signal_count"].values[keep_mask]
for i in range(len(oof_final)):
    prob = oof_final[i]
    signals = train_signals[i]
    if prob > 0.3 and signals < 2:
        oof_final[i] = prob * 0.6
    elif prob > 0.3 and signals < 3:
        oof_final[i] = prob * 0.8

best_f1, best_f2 = 0, 0
best_t_f1, best_t_f2 = 0.5, 0.5
for t in np.arange(0.1, 0.95, 0.005):
    p = (oof_final > t).astype(int)
    if p.sum() == 0: continue
    f1 = f1_score(y_clean, p)
    f2 = fbeta_score(y_clean, p, beta=2)
    if f1 > best_f1: best_f1, best_t_f1 = f1, t
    if f2 > best_f2: best_f2, best_t_f2 = f2, t

print(f"F1-optimal: t={best_t_f1:.3f} F1={best_f1:.4f}")
print(f"F2-optimal: t={best_t_f2:.3f} F2={best_f2:.4f}")

# Use F2 threshold (same as Phase3/V4 which had best F1 on test)
final_threshold = best_t_f2
preds = (oof_final > final_threshold).astype(int)
cm = confusion_matrix(y_clean, preds)
print(f"\nUsing t={final_threshold:.3f}:")
print(f"  F1={f1_score(y_clean,preds):.4f}  F2={fbeta_score(y_clean,preds,beta=2):.4f}")
print(f"  P={precision_score(y_clean,preds):.4f}  R={recall_score(y_clean,preds):.4f}")
print(f"  CM: TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}")

test["is_mule_prob"] = t_final
mule_preds = test[test["is_mule_prob"] > final_threshold]["account_id"].tolist()
print(f"\nPredicted mules on TEST: {len(mule_preds)}")

F1-optimal: t=0.415 F1=0.8639
F2-optimal: t=0.300 F2=0.9203

Using t=0.300:
  F1=0.8542  F2=0.9203
  P=0.7627  R=0.9705
  CM: TN=92,723 FP=685 FN=67 TP=2,202

Predicted mules on TEST: 1783


## 8 — Adaptive CDF Temporal Windows (V6 style)

In [9]:
print("=" * 60)
print("ADAPTIVE CDF TEMPORAL WINDOWS")
print("=" * 60)

parts = sorted(glob(f"{DATA_DIR}/transactions/batch-*/part_*.parquet"))
print(f"Transaction parts: {len(parts)}")

temporal_threshold = final_threshold * 0.25
high_prob_ids = set(test[test["is_mule_prob"] > temporal_threshold]["account_id"].tolist())
print(f"Accounts for temporal: {len(high_prob_ids):,}")

daily_vol = {}
for i, p in enumerate(parts):
    try:
        ds = pq.read_table(p, columns=["account_id", "transaction_timestamp", "amount"],
                           filters=[("account_id", "in", list(high_prob_ids))])
        df = ds.to_pandas()
    except:
        df = pd.read_parquet(p, columns=["account_id", "transaction_timestamp", "amount"])
        df = df[df["account_id"].isin(high_prob_ids)]
    if df.empty: continue
    df["ts"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    df["date"] = df["ts"].dt.date
    df["abs_amount"] = df["amount"].abs()
    grp = df.groupby(["account_id", "date"])["abs_amount"].sum()
    for (aid, dt), vol in grp.items():
        if aid not in daily_vol:
            daily_vol[aid] = {}
        daily_vol[aid][dt] = daily_vol[aid].get(dt, 0) + vol
    del df
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(parts)}] processed")
    gc.collect()

print(f"Built series for {len(daily_vol):,} accounts")

ADAPTIVE CDF TEMPORAL WINDOWS
Transaction parts: 396
Accounts for temporal: 2,855
  [100/396] processed
  [200/396] processed
  [300/396] processed
Built series for 2,843 accounts


In [10]:
def v7_temporal_window(vol_dict):
    """V7: Multi-scale densest window + inner CDF trimming."""
    if len(vol_dict) < 3:
        return "", ""

    dates = sorted(vol_dict.keys())
    vols = np.array([vol_dict[d] for d in dates])
    total = vols.sum()
    if total == 0:
        return "", ""

    # Try multiple window sizes, pick tightest with 50%+ coverage
    best_start, best_end = 0, len(dates) - 1
    for window_days in [14, 30, 60, 90]:
        best_wvol = 0
        b_s, b_e = 0, 0
        for j in range(len(vols)):
            k = j
            wvol = 0
            while k < len(vols) and (dates[k] - dates[j]).days <= window_days:
                wvol += vols[k]
                k += 1
            if wvol > best_wvol:
                best_wvol = wvol
                b_s, b_e = j, k - 1
        if best_wvol / total >= 0.50:
            best_start, best_end = b_s, b_e
            break

    # Inner CDF trim (5%-95%)
    w_vols = vols[best_start:best_end+1]
    w_dates = dates[best_start:best_end+1]
    if len(w_vols) > 5:
        w_cum = np.cumsum(w_vols)
        w_tot = w_cum[-1]
        if w_tot > 0:
            w_cdf = w_cum / w_tot
            s_idx = min(np.searchsorted(w_cdf, 0.05), len(w_dates) - 1)
            e_idx = min(np.searchsorted(w_cdf, 0.95), len(w_dates) - 1)
            w_dates = w_dates[s_idx:e_idx+1]

    if len(w_dates) == 0:
        w_dates = dates[best_start:best_end+1]

    return f"{w_dates[0]}T00:00:00", f"{w_dates[-1]}T23:59:59"

temporal_windows = {}
for aid in high_prob_ids:
    if aid in daily_vol:
        s, e = v7_temporal_window(daily_vol[aid])
        temporal_windows[aid] = (s, e)
    else:
        temporal_windows[aid] = ("", "")

n_with = sum(1 for s, e in temporal_windows.values() if s != "")
print(f"Accounts with windows: {n_with:,}/{len(high_prob_ids):,}")

widths = []
for s, e in temporal_windows.values():
    if s and e:
        widths.append((pd.to_datetime(e) - pd.to_datetime(s)).days)
wa = np.array(widths) if widths else np.array([0])
print(f"Window width: median={np.median(wa):.0f}d, mean={wa.mean():.0f}d, "
      f"p25={np.percentile(wa,25):.0f}d, p75={np.percentile(wa,75):.0f}d")

Accounts with windows: 2,842/2,855
Window width: median=500d, mean=535d, p25=52d, p75=766d


## 9 — Generate Submission

In [11]:
print("=" * 60)
print("GENERATING SUBMISSION V7")
print("=" * 60)

submission = pd.DataFrame({
    "account_id": test["account_id"],
    "is_mule": test["is_mule_prob"],
    "suspicious_start": "",
    "suspicious_end": ""
})

for aid, (s, e) in temporal_windows.items():
    mask = submission["account_id"] == aid
    submission.loc[mask, "suspicious_start"] = s
    submission.loc[mask, "suspicious_end"] = e

submission.to_csv("submission_v7.csv", index=False)

# Sanity checks
print(f"Submission: {submission.shape}")
print(f"  Mean prob:    {submission['is_mule'].mean():.4f}")
print(f"  >50% mule:    {(submission['is_mule']>0.5).sum():,}")
print(f"  >30% mule:    {(submission['is_mule']>0.3).sum():,}")
print(f"  >80% mule:    {(submission['is_mule']>0.8).sum():,}")
print(f"  With windows: {(submission['suspicious_start']!='').sum():,}")

# Sanity: compare to Phase3/V4
print(f"\n⚠️ Calibration check:")
print(f"  Expected mule rate: ~2.8% → ~{int(len(test)*0.028):,} mules")
print(f"  Predicted >0.5:     {(submission['is_mule']>0.5).sum():,}")
print(f"  If >2× expected, check for calibration issues")

print(f"\n✅ submission_v7.csv saved")
print(f"Total: {time.time()-t0:.0f}s = {(time.time()-t0)/60:.1f} min")

GENERATING SUBMISSION V7
Submission: (64062, 4)
  Mean prob:    0.0226
  >50% mule:    1,497
  >30% mule:    1,783
  >80% mule:    215
  With windows: 2,842

⚠️ Calibration check:
  Expected mule rate: ~2.8% → ~1,793 mules
  Predicted >0.5:     1,497
  If >2× expected, check for calibration issues

✅ submission_v7.csv saved
Total: 293s = 4.9 min
